In [ ]:
import sys,os
hsaDir = os.path.abspath("../../")
#srcDir = os.path.join(hsaDir,"src")
#sys.path.insert(0,srcDir)

In [ ]:
from heliostat_surface_analysis.loader.scaling_loader import ScalingLoader
from heliostat_surface_analysis.loader.default_loader import DefaultLoader
from heliostat_surface_analysis.preprocessor.basic_preprocessor import BasicPreprocessor
from heliostat_surface_analysis.preprocessor.no_preprocessor import NoPreprocessor
from heliostat_surface_analysis.mesher.delaunay_mesher import DelaunayMesher
from heliostat_surface_analysis.mesher.bspline_mesher import BSplineMesher
from heliostat_surface_analysis.analyser.fit_surface_analyser import FitSurfaceAnalyser
from heliostat_surface_analysis.analyser.fixed_focal_length_analyser import FixedFocalLengthAnalyser
from heliostat_surface_analysis.aggregator.aggregator import WeightedRMSAggregator
from heliostat_surface_analysis.analysis_manager import AnalysisManager
from heliostat_surface_analysis.plotting.plotting import plot_2d_surface,set_plot_font_size

In [ ]:
import pandas as pd
import progressbar
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
%matplotlib inline

In [ ]:
# for progressbar
widgets = [' [',
         progressbar.Timer(format= 'elapsed time: %(elapsed)s'),
         '] ',
           progressbar.Bar('*'),' (',
           progressbar.ETA(), ') ',
          ]

### Setup analysis components

In [ ]:
preprocessor = BasicPreprocessor()
meshers = {}
# NOTE: setting the max_aspect_ratio is important
meshers["Delaunay"] = DelaunayMesher({'max_aspect_ratio':10})
meshers["B-spline"] = BSplineMesher({'polynomial_degree':3,'dx':0.01,'dy':0.01})
analysers = {}
analysers["FitSurface"] = FitSurfaceAnalyser({"surface":"circular_paraboloid"})
aggregator = WeightedRMSAggregator()

In [ ]:
# currently only "PEG Field2 scans" and "circular paraboid validation" implemented, others can be added below
scansCase = "PEG Field2 scans"

In [ ]:
if scansCase=="PEG Field2 scans":
    loader = ScalingLoader({"units":"mm"})
    scansDir = os.path.join(hsaDir,"examples","command_line","circular_paraboloid_peg","scans")
    designFocalLengthsFile = os.path.join(hsaDir,"examples","command_line","circular_paraboloid_peg","design_focal_lengths.txt")
    analysers["FixedFocalLength"] = FixedFocalLengthAnalyser({"focal_length_file":designFocalLengthsFile})
elif scansCase=="circular paraboid validation":
    loader = DefaultLoader()
    scansDir = os.path.join(hsaDir,"examples","command_line","ideal_circular_paraboloid_random","scans")
    designFocalLengthsFile = os.path.join(hsaDir,"examples","command_line","ideal_circular_paraboloid_random","design_focal_lengths.txt")
    analysers["FixedFocalLength"] = FixedFocalLengthAnalyser({"focal_length":29.})
else:
    raise RuntimeError("not recognised")

In [ ]:
scanFiles = sorted(os.listdir(scansDir))

### Compare meshers

In [ ]:
analyser = analysers["FitSurface"]
scanNames = []
focalLengths = []
slopeErrors = {}
for mesherName,_ in meshers.items():
    slopeErrors[mesherName] = []
bar = progressbar.ProgressBar(maxval=len(scanFiles)).start()
for i,scanFile in enumerate(scanFiles):
    scanPath = os.path.join(scansDir,scanFile)
    for mesherName,mesher in meshers.items():
        result = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=False,write_stls=False)
        slopeErrors[mesherName].append(result["aggregate"])
        focalLength = 1/4./result["coeff: a(x^2 + y^2)"]
        scanName = result["name"]
    # these are the same irrespective of what mesher is used
    scanNames.append(scanName)
    focalLengths.append(focalLength)
    bar.update(i+1)
bar.finish()

In [ ]:
dataDict = {}
dataDict["scan"] = scanNames
dataDict["f"] = focalLengths
for mesherName,mesher in meshers.items():
    dataDict[f"sigma ({mesherName})"] = slopeErrors[mesherName]
df = pd.DataFrame(dataDict)
df.set_index("scan",inplace=True)
df.head(n=150)

### Compare analysers

In [ ]:
mesher = meshers["Delaunay"]
scanNames = []
slopeErrors = {}
focalLengths = {}
for analyserName,_ in analysers.items():
    slopeErrors[analyserName] = []
    focalLengths[analyserName] = []
bar = progressbar.ProgressBar(maxval=len(scanFiles)).start()
for i,scanFile in enumerate(scanFiles):
    scanPath = os.path.join(scansDir,scanFile)
    for analyserName,analyser in analysers.items():
        result = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=False,write_stls=False)
        slopeErrors[analyserName].append(result["aggregate"])
        scanName = result["name"]
        if analyserName=="FitSurface":
            f = 1/4./result["coeff: a(x^2 + y^2)"]
        else:
            f = analyser.focal_length_model.get(scanName)
        focalLengths[analyserName].append(f)
    # these are the same irrespective of what analyser is used
    scanNames.append(scanName)
    bar.update(i+1)
bar.finish()

In [ ]:
dataDict = {}
dataDict["scan"] = scanNames
for analyserName,analyser in analysers.items():
    dataDict[f"f ({analyserName})"] = focalLengths[analyserName]
    dataDict[f"sigma ({analyserName})"] = slopeErrors[analyserName]
df = pd.DataFrame(dataDict)
df.set_index("scan",inplace=True)
df.head(n=150)

###  plotting of slope error spatial distribution

In [ ]:
scanFile = scanFiles[1]
scanPath = os.path.join(scansDir,scanFile)

In [ ]:
analyser = analysers["FitSurface"]

### 1. via the analyse_scan function (quick and easy but not very flexible)

In [ ]:
for mesherName,mesher in meshers.items():
    print(mesherName)
    _ = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=True,write_stls=False)


### 2. via custom code (allows modification of figure size and data range etc)

In [ ]:
for mesherName,mesher in meshers.items():
    scan_name = os.path.splitext(os.path.basename(scanPath))[0]
    raw_x, raw_y, raw_z = loader.load_points(scanPath)
    x, y, z = preprocessor.preprocess_points(raw_x, raw_y, raw_z)
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    f,ax = plt.subplots()
    f.set_size_inches(12,10)
    ax.set_title(mesherName)
    plot_2d_surface(mesh,slope_errors,label="slope error (rad)",ax=ax,fontsize=16,cbxy=(4.0,0.29))
    f,ax = plt.subplots()
    f.set_size_inches(10,8)
    ax.set_title(mesherName)
    ax.hist(slope_errors,bins=20)
    ax.grid()
    ax.set_xlabel("slope error (rad)")
    set_plot_font_size(ax,16,False)

### plotting of slope error distribution for the scanned facet population

In [ ]:
if len(scanFiles)>25:
    f,ax = plt.subplots()
    f.set_size_inches(10,8)
    for key,value in slopeErrors.items():
        ax.hist(x=value, bins=int(len(scanFiles)/10.), alpha=0.5, label=key, rwidth=0.85)
    ax.grid(axis='y', alpha=0.75)
    ax.set_xlabel('Slope error, $\sigma$ (mrad)')
    ax.set_ylabel('Frequency')
    ax.legend()
    #maxfreq = n.max()
    # Set a clean upper y-axis limit.
    #plt.ylim(ymax=np.ceil(maxfreq / 10) * 10 if maxfreq % 10 else maxfreq + 10)
    plt.show()

### check sensitivity to aspect ratio

In [ ]:
scan_name = os.path.splitext(os.path.basename(scanPath))[0]
raw_x, raw_y, raw_z = loader.load_points(scanPath)
x, y, z = preprocessor.preprocess_points(raw_x, raw_y, raw_z)
max_aspect_ratios = list(np.arange(2,10,1)) + list(np.arange(10,100,10)) + list(np.arange(100,1000,100))
aggregated_slope_errors = []
triangle_counts = []
for max_aspect_ratio in max_aspect_ratios:
    mesher = DelaunayMesher({"max_aspect_ratio":max_aspect_ratio})
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    triangle_counts.append(len(areas))
    aggregate = aggregator.aggregate(slope_errors,areas)
    aggregated_slope_errors.append(aggregate)
f,ax = plt.subplots()
ax2 = plt.twinx(ax)
f.set_size_inches(10,8)
ax.plot(max_aspect_ratios,aggregated_slope_errors)
ax2.plot(max_aspect_ratios,triangle_counts,color="r")
ax.set_xlabel("max. aspect ratio")
ax.set_ylabel("aggregated slope error (rad)")
ax2.set_ylabel("triangle counts")
ax2.yaxis.label.set_color('red')
ax2.tick_params(axis='y', colors='red')
set_plot_font_size(ax,16,False)
set_plot_font_size(ax2,16,False)
ax.grid()

In [ ]:
scan_name = os.path.splitext(os.path.basename(scanPath))[0]
raw_x, raw_y, raw_z = loader.load_points(scanPath)
x, y, z = preprocessor.preprocess_points(raw_x, raw_y, raw_z)
max_aspect_ratios = list(np.arange(2,10,1)) + list(np.arange(10,100,10))
aggregated_slope_errors = []
triangle_counts = []
for max_aspect_ratio in max_aspect_ratios:
    mesher = DelaunayMesher({"max_aspect_ratio":max_aspect_ratio})
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    triangle_counts.append(len(areas))
    aggregate = aggregator.aggregate(slope_errors,areas)
    aggregated_slope_errors.append(aggregate)
f,ax = plt.subplots()
ax2 = plt.twinx(ax)
f.set_size_inches(10,8)
ax.plot(max_aspect_ratios,aggregated_slope_errors)
ax2.plot(max_aspect_ratios,triangle_counts,color="r")
ax.set_xlabel("max. aspect ratio")
ax.set_ylabel("aggregated slope error (rad)")
ax2.set_ylabel("triangle counts")
ax2.yaxis.label.set_color('red')
ax2.tick_params(axis='y', colors='red')
set_plot_font_size(ax,16,False)
set_plot_font_size(ax2,16,False)
ax.grid()

In [ ]:
scan_name = os.path.splitext(os.path.basename(scanPath))[0]
raw_x, raw_y, raw_z = loader.load_points(scanPath)
x, y, z = preprocessor.preprocess_points(raw_x, raw_y, raw_z)
max_aspect_ratios = list(np.arange(2,10,1))
aggregated_slope_errors = []
triangle_counts = []
for max_aspect_ratio in max_aspect_ratios:
    mesher = DelaunayMesher({"max_aspect_ratio":max_aspect_ratio})
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    triangle_counts.append(len(areas))
    aggregate = aggregator.aggregate(slope_errors,areas)
    aggregated_slope_errors.append(aggregate)
f,ax = plt.subplots()
ax2 = plt.twinx(ax)
f.set_size_inches(10,8)
ax.plot(max_aspect_ratios,aggregated_slope_errors)
ax2.plot(max_aspect_ratios,triangle_counts,color="r")
ax.set_xlabel("max. aspect ratio")
ax.set_ylabel("aggregated slope error (rad)")
ax2.set_ylabel("triangle counts")
ax2.yaxis.label.set_color('red')
ax2.tick_params(axis='y', colors='red')
set_plot_font_size(ax,16,False)
set_plot_font_size(ax2,16,False)
ax.grid()